# Retrospect Model Panel Review

This notebook is for inspecting:
- rough heuristic archive cost estimates
- empirical 3-chat panel runs
- manual quality review scores
- privacy/data-retention review notes

Expected inputs live under ignored runtime directories in `retrospect/data/`.

In [ ]:
from pathlib import Path
import csv
import json
import statistics

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print('matplotlib not installed; tables will still work.')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RETROSPECT = ROOT / 'retrospect' if (ROOT / 'retrospect').exists() else ROOT
DATA = RETROSPECT / 'data'
CATALOG = RETROSPECT / 'config' / 'model_catalog.json'
REPORT = DATA / 'reports' / 'model-cost-analysis.md'
TRIO = DATA / 'samples' / 'model-eval-trio.json'
EVALUATIONS = DATA / 'evaluations'


In [ ]:
catalog = json.loads(CATALOG.read_text(encoding='utf-8'))
trio = json.loads(TRIO.read_text(encoding='utf-8')) if TRIO.exists() else None

print('Catalog models:', len(catalog['models']))
if trio:
    print('Eval trio chats:')
    for item in trio['selected_chats']:
        print(f"  {item['label']}: {item['relative_path']} ({item['byte_size']} bytes)")


In [ ]:
def latest_bundle():
    bundles = sorted(EVALUATIONS.glob('*__model-panel-trio'))
    return bundles[-1] if bundles else None

bundle = latest_bundle()
if bundle:
    print('Using bundle:', bundle)
    bundle_manifest = json.loads((bundle / 'bundle_manifest.json').read_text(encoding='utf-8'))
else:
    bundle_manifest = None
    print('No evaluation bundle found yet. Run scripts/run_model_panel.py first.')


In [ ]:
def load_csv_rows(path):
    if not path.exists():
        return []
    with open(path, newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

quality_rows = load_csv_rows(bundle / 'quality_scores.csv') if bundle else []
privacy_rows = load_csv_rows(bundle / 'privacy_review.csv') if bundle else []

print('Quality rows:', len(quality_rows))
print('Privacy rows:', len(privacy_rows))


In [ ]:
def numeric(values):
    parsed = []
    for value in values:
        try:
            parsed.append(float(value))
        except (TypeError, ValueError):
            pass
    return parsed

def summarize_quality(rows):
    by_model = {}
    for row in rows:
        bucket = by_model.setdefault(row['model'], {'factual_accuracy': [], 'evidence_quality': [], 'false_positive_risk': [], 'completeness': [], 'synthesis_utility': []})
        for key in bucket:
            bucket[key].extend(numeric([row.get(key)]))
    summary = {}
    for model, metrics in by_model.items():
        summary[model] = {k: (statistics.mean(v) if v else None) for k, v in metrics.items()}
    return summary

quality_summary = summarize_quality(quality_rows)
quality_summary


In [ ]:
if bundle_manifest:
    empirical = []
    for result in bundle_manifest['run_results']:
        manifest_path = result.get('manifest_path')
        if not manifest_path:
            continue
        data = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
        empirical.append({
            'model': result['model'],
            'reported_cost_total': data.get('reported_cost_total'),
            'actual_prompt_tokens': data.get('actual_prompt_tokens'),
            'actual_completion_tokens': data.get('actual_completion_tokens'),
            'actual_total_tokens': data.get('actual_total_tokens'),
        })
    empirical
else:
    []


In [ ]:
if plt and bundle_manifest:
    rows = []
    for result in bundle_manifest['run_results']:
        manifest_path = result.get('manifest_path')
        if not manifest_path:
            continue
        data = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
        if data.get('reported_cost_total') is not None:
            rows.append((result['model'], data['reported_cost_total']))
    rows.sort(key=lambda item: item[1])
    plt.figure(figsize=(12, 5))
    plt.bar([r[0] for r in rows], [r[1] for r in rows])
    plt.xticks(rotation=75, ha='right')
    plt.ylabel('USD')
    plt.title('Empirical 3-chat panel cost by model')
    plt.tight_layout()
    plt.show()


## Decision Matrix Guidance

Populate the final decision artifact only after:
- empirical 3-chat runs exist for all shortlisted models
- quality review CSV is filled in
- privacy review CSV is filled in

Recommended decision criteria:
- Pass 1 factual extraction quality
- Pass 2 projects/goals quality
- Pass 3 people/values quality
- Pass 4 psych quality
- Empirical 3-chat cost
- Projected archive cost
- Privacy/data retention posture
- Availability/routing confidence
